# Clase 086 — Feature importance

Medimos qué variables aportan a un modelo de árboles, distinguiendo
**MDI** (`feature_importances_`, sesgado por cardinalidad y calculado en train) de
**permutation importance** (sobre test, model-agnostic). Demostramos el sesgo de MDI
con una feature espuria de alta cardinalidad.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

## 1. Dataset `load_breast_cancer` con nombres de features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

np.random.seed(42)

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape)

## 2. MDI: `feature_importances_` del Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
rf.fit(X_train, y_train)
print(f'acc test: {rf.score(X_test, y_test):.4f}')

mdi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
assert abs(mdi.sum() - 1.0) < 1e-6, 'MDI está normalizado (suma 1)'
print('\nTop-5 MDI:')
print(mdi.head().round(4).to_string())

## 3. Permutation importance sobre test (`n_repeats=10`)

In [ ]:
perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=1)
perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print('Top-5 permutation:')
print(perm_imp.head().round(4).to_string())

comp = pd.DataFrame({'MDI': mdi, 'permutation': perm_imp}).sort_values('MDI', ascending=False)
print('\nComparativa (top-8):')
print(comp.head(8).round(4).to_string())

## 4. Comparativa visual MDI vs permutation (top-10)

In [ ]:
top = mdi.head(10).index[::-1]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axes[0].barh(top, mdi[top].values, color='#37a')
axes[0].set_title('MDI (train, normalizado)')
axes[1].barh(top, perm_imp[top].values, color='#3a7')
axes[1].set_title('Permutation (test)')
for ax in axes:
    ax.set_xlabel('importancia')
plt.tight_layout()
plt.show()

## 5. Sesgo de MDI: feature espuria de alta cardinalidad

Agregamos un `random_id` sin poder predictivo. MDI le asigna importancia inflada
(muchos splits candidatos); permutation la ignora.

In [ ]:
X_train2 = X_train.copy()
X_test2 = X_test.copy()
rng = np.random.default_rng(42)
X_train2['random_id'] = rng.random(len(X_train2))
X_test2['random_id']  = rng.random(len(X_test2))

rf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
rf2.fit(X_train2, y_train)

mdi2 = pd.Series(rf2.feature_importances_, index=X_train2.columns)
perm2 = permutation_importance(
    rf2, X_test2, y_test, n_repeats=10, random_state=42, n_jobs=1)
perm2 = pd.Series(perm2.importances_mean, index=X_train2.columns)

print(f'random_id  MDI         : {mdi2["random_id"]:.4f}  (inflado, tiene señal espuria)')
print(f'random_id  permutation : {perm2["random_id"]:.4f}  (~0, correctamente ignorado)')
assert mdi2['random_id'] > perm2['random_id'], 'MDI infla la feature espuria vs permutation'
print('assert OK: MDI sobreestima la feature de alta cardinalidad; permutation no.')

## Ejercicios

1. Reportá el top-5 de features según MDI y según permutation. ¿Coincide el top-3?
   Justificá las diferencias por cardinalidad/correlación.
2. Detectá pares de features correlacionadas con `X.corr()` y explicá cómo eso afecta
   a la permutation importance.
3. Repetí el análisis con `GradientBoostingClassifier` y compará los rankings.
4. Usá `SelectFromModel(rf, threshold='median')` para quedarte con las features
   importantes y reentrená; ¿cae mucho la accuracy?

## Conclusiones

- MDI es rápido pero sesgado hacia features de **alta cardinalidad** y se calcula en train.
- Permutation importance se mide sobre datos **no vistos** y es model-agnostic; más
  honesto para auditar.
- Ambos flaquean con features **correlacionadas** (reparten crédito arbitrariamente).
- Regla práctica: rankeá con permutation sobre test, no con MDI sobre train.